# 03 - Match Best Picture nominees to MovieLens movies

Goal: give each Best Picture nominee of the ceremonies 1996 to 2017 its MovieLens `movieId`, so that we can read its ratings later.

Method: the IMDb id of each nominee (`FilmId` in `full_data.csv`, with the prefix `tt`) is compared with `imdbId` in MovieLens `links.csv` (without the prefix). The match is then checked by comparing titles and years side by side.

Output: `data/processed/best_picture_nominees_1996_2017.csv`.

In [1]:
import duckdb

con = duckdb.connect()

In [2]:
# full_data.csv: tab-separated, and rows have a variable number of fields (empty trailing fields are missing),
# so we pad the missing trailing fields with NULL. All columns are read as text.
con.sql("""
CREATE VIEW oscars_full AS
SELECT * FROM read_csv('../data/raw/oscar_archive_1927_2025/full_data.csv',
                       delim='\t', quote='', escape='', header=true, all_varchar=true, null_padding=true)
""")

In [3]:
# Discovery: MovieLens links, movieId <-> imdbId (imdbId has no 'tt' prefix)
con.sql("SELECT * FROM '../data/raw/ml-32m/links.csv' LIMIT 5")

┌─────────┬─────────┬────────┐
│ movieId │ imdbId  │ tmdbId │
│  int64  │ varchar │ int64  │
├─────────┼─────────┼────────┤
│       1 │ 0114709 │    862 │
│       2 │ 0113497 │   8844 │
│       3 │ 0113228 │  15602 │
│       4 │ 0114885 │  31357 │
│       5 │ 0113041 │  11862 │
└─────────┴─────────┴────────┘

In [4]:
# Discovery: Best Picture rows in full_data.csv for two ceremonies (FilmId has the 'tt' prefix)
con.sql("SELECT Ceremony, Year, Film, FilmId, Winner FROM oscars_full WHERE CanonicalCategory = 'BEST PICTURE' AND Ceremony IN ('70', '71') ORDER BY Ceremony, Film")

┌──────────┬─────────┬─────────────────────┬───────────┬─────────┐
│ Ceremony │  Year   │        Film         │  FilmId   │ Winner  │
│ varchar  │ varchar │       varchar       │  varchar  │ varchar │
├──────────┼─────────┼─────────────────────┼───────────┼─────────┤
│ 70       │ 1997    │ As Good as It Gets  │ tt0119822 │ NULL    │
│ 70       │ 1997    │ Good Will Hunting   │ tt0119217 │ NULL    │
│ 70       │ 1997    │ L.A. Confidential   │ tt0119488 │ NULL    │
│ 70       │ 1997    │ The Full Monty      │ tt0119164 │ NULL    │
│ 70       │ 1997    │ Titanic             │ tt0120338 │ True    │
│ 71       │ 1998    │ Elizabeth           │ tt0127536 │ NULL    │
│ 71       │ 1998    │ Life Is Beautiful   │ tt0118799 │ NULL    │
│ 71       │ 1998    │ Saving Private Ryan │ tt0120815 │ NULL    │
│ 71       │ 1998    │ Shakespeare in Love │ tt0138097 │ True    │
│ 71       │ 1998    │ The Thin Red Line   │ tt0120863 │ NULL    │
├──────────┴─────────┴─────────────────────┴───────────┴──────

In [5]:
# Step 1: find Titanic's movieId in MovieLens from its IMDb id (tt0120338 without the "tt")
con.sql("SELECT * FROM '../data/raw/ml-32m/links.csv' WHERE imdbId = '0120338'")

┌─────────┬─────────┬────────┐
│ movieId │ imdbId  │ tmdbId │
│  int64  │ varchar │ int64  │
├─────────┼─────────┼────────┤
│    1721 │ 0120338 │    597 │
└─────────┴─────────┴────────┘

In [6]:
# Step 2: check that movieId 1721 is really Titanic (title in movies.csv)
con.sql("SELECT * FROM '../data/raw/ml-32m/movies.csv' WHERE movieID = 1721")

┌─────────┬────────────────┬───────────────┐
│ movieId │     title      │    genres     │
│  int64  │    varchar     │    varchar    │
├─────────┼────────────────┼───────────────┤
│    1721 │ Titanic (1997) │ Drama|Romance │
└─────────┴────────────────┴───────────────┘

In [7]:
# Step 3: link the 10 Best Picture nominees of ceremonies 70 and 71 to their MovieLens movieId
con.sql("SELECT o.Film, o.FilmId, l.movieId FROM oscars_full o LEFT JOIN (SELECT movieId, 'tt' || imdbId AS FilmId FROM '../data/raw/ml-32m/links.csv') l ON o.FilmId = l.FilmId WHERE o.CanonicalCategory = 'BEST PICTURE' AND o.Ceremony IN ('70', '71') ORDER BY o.Ceremony, o.Film")

┌─────────────────────┬───────────┬─────────┐
│        Film         │  FilmId   │ movieId │
│       varchar       │  varchar  │  int64  │
├─────────────────────┼───────────┼─────────┤
│ As Good as It Gets  │ tt0119822 │    1784 │
│ Good Will Hunting   │ tt0119217 │    1704 │
│ L.A. Confidential   │ tt0119488 │    1617 │
│ The Full Monty      │ tt0119164 │    1641 │
│ Titanic             │ tt0120338 │    1721 │
│ Elizabeth           │ tt0127536 │    2336 │
│ Life Is Beautiful   │ tt0118799 │    2324 │
│ Saving Private Ryan │ tt0120815 │    2028 │
│ Shakespeare in Love │ tt0138097 │    2396 │
│ The Thin Red Line   │ tt0120863 │    2427 │
├─────────────────────┴───────────┴─────────┤
│ 10 rows                         3 columns │
└───────────────────────────────────────────┘

In [8]:
# Step 4: coverage for all Best Picture nominees of ceremonies 68 to 89 (1996 to 2017)
con.sql("SELECT COUNT(*) AS nb_nominees, COUNT(l.movieId) AS nb_matched FROM oscars_full o LEFT JOIN (SELECT movieId, 'tt' || imdbId AS FilmId FROM '../data/raw/ml-32m/links.csv') l ON o.FilmId = l.FilmId WHERE o.CanonicalCategory = 'BEST PICTURE' AND CAST(o.Ceremony AS INTEGER) BETWEEN 68 AND 89")

┌─────────────┬────────────┐
│ nb_nominees │ nb_matched │
│    int64    │   int64    │
├─────────────┼────────────┤
│         142 │        142 │
└─────────────┴────────────┘

In [9]:
# Step 5: side-by-side check, Oscars title/year vs MovieLens title (142 rows)
con.sql("SELECT o.Ceremony, o.Film AS oscar_title, o.Year AS film_year, l.movieId, m.title AS movielens_title FROM oscars_full o LEFT JOIN (SELECT movieId, 'tt' || imdbId AS FilmId FROM '../data/raw/ml-32m/links.csv') l ON o.FilmId = l.FilmId LEFT JOIN '../data/raw/ml-32m/movies.csv' m ON l.movieId = m.movieId WHERE o.CanonicalCategory = 'BEST PICTURE' AND CAST(o.Ceremony AS INTEGER) BETWEEN 68 AND 89 ORDER BY CAST(o.Ceremony AS INTEGER), o.Film").show(max_rows=150)


┌──────────┬───────────────────────────────────────────────────┬───────────┬─────────┬───────────────────────────────────────────────────────────┐
│ Ceremony │                    oscar_title                    │ film_year │ movieId │                      movielens_title                      │
│ varchar  │                      varchar                      │  varchar  │  int64  │                          varchar                          │
├──────────┼───────────────────────────────────────────────────┼───────────┼─────────┼───────────────────────────────────────────────────────────┤
│ 68       │ Apollo 13                                         │ 1995      │     150 │ Apollo 13 (1995)                                          │
│ 68       │ Babe                                              │ 1995      │      34 │ Babe (1995)                                               │
│ 68       │ Braveheart                                        │ 1995      │     110 │ Braveheart (1995)              

In [10]:
# Step 6: save the 142 Best Picture nominees (ceremonies 1996-2017) with their MovieLens movieId
con.sql("COPY (SELECT CAST(o.Year AS INTEGER) + 1 AS ceremony_year, CAST(o.Year AS INTEGER) AS film_year, o.Film AS title, COALESCE(o.Winner = 'True', false) AS is_winner, l.movieId FROM oscars_full o LEFT JOIN (SELECT movieId, 'tt' || imdbId AS FilmId FROM '../data/raw/ml-32m/links.csv') l ON o.FilmId = l.FilmId WHERE o.CanonicalCategory = 'BEST PICTURE' AND CAST(o.Ceremony AS INTEGER) BETWEEN 68 AND 89 ORDER BY ceremony_year, is_winner DESC, title) TO '../data/processed/best_picture_nominees_1996_2017.csv' (HEADER, DELIMITER ',')")

In [11]:
# Step 7: check the saved file (142 rows, 22 winners, a movieId for every row, ceremonies 1996 to 2017)
con.sql("SELECT COUNT(*) AS n_rows, SUM(CASE WHEN is_winner THEN 1 ELSE 0 END) AS n_winners, COUNT(movieId) AS n_with_movieId, MIN(ceremony_year) AS first_year, MAX(ceremony_year) AS last_year FROM '../data/processed/best_picture_nominees_1996_2017.csv'")

┌────────┬───────────┬────────────────┬────────────┬───────────┐
│ n_rows │ n_winners │ n_with_movieId │ first_year │ last_year │
│ int64  │  int128   │     int64      │   int64    │   int64   │
├────────┼───────────┼────────────────┼────────────┼───────────┤
│    142 │        22 │            142 │       1996 │      2017 │
└────────┴───────────┴────────────────┴────────────┴───────────┘

## What we learned
- The 142 Best Picture nominees of ceremonies 1996 to 2017 are all found in MovieLens through their IMDb id (`FilmId` = `'tt' || imdbId`), with no duplicated row.
- Titles and years were compared side by side: consistent.
- Nominees were checked against the official Academy database for ceremonies 1996 to 2015, and against Wikipedia for 2016 and 2017.
- Output: `data/processed/best_picture_nominees_1996_2017.csv`. Next: `04_yearly_ratings.ipynb`.